# 🎯 4. Closures

A **closure** is a function that remembers variables from its enclosing scope, even after that scope has finished executing.

Closures are the **mechanism** that makes decorators work.

| ☕ Java | 🐍 Python |
|---------|----------|
| Lambda captures effectively-final variables | Functions capture **any** variables |
| Can't modify captured variables | `nonlocal` keyword allows mutation |
| Anonymous inner classes | Closures are regular functions with memory |

In [1]:
from typing import Callable
from functools import partial

## 4.1 What is a Closure?

When an inner function references a variable from its enclosing function, and the outer function returns the inner function — that's a closure.

In [ ]:
def outer_func():
    x = 10

    def inner_func():
        return x
    
    return inner_func

f = outer_func()

print(f())

10


In [ ]:
def make_greeter(greeting: str) -> Callable[[str], str]:
    """Outer function - creates a closure"""

    def greeter(name: str) -> str:
        """Inner function - the closure itself"""
        return f"{greeting}, {name}"
    
    return greeter

say_hello = make_greeter("Hello")
say_ola = make_greeter("Ola")

print(say_hello("Alice")) # Hello, Alice
print(say_ola("Bob")) # Ola, Bob


Hello, Alice
Ola, Bob


## 4.2 Factory Functions

Closures are perfect for creating **families of functions** from a template.

In [6]:
def make_multiplier(factor: int) -> Callable[[int], int]:
    """Factory multiplier"""

    def multiplier(x: int) -> int:
        return x * factor
    
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
times_ten = make_multiplier(10)

print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")
print(f"times_ten(10) = {times_ten(10)}")

double(5) = 10
triple(5) = 15
times_ten(10) = 100


In [7]:
def make_power(exp: int) -> Callable[[int], int]:

    def power(base: int) -> int:
        return base ** exp
    return power

square = make_power(2)
cube = make_power(3)
print(f"square(10) = {square(10)}")
print(f"cube(10) = {cube(10)}")

square(10) = 100
cube(10) = 1000


## 4.3 `nonlocal` — Modifying Captured Variables

By default, you can **read** captured variables but not **reassign** them.
Without `nonlocal`, Python treats the assignment as a **new local variable**.

| Keyword | Scope |
|---------|-------|
| `global` | Module-level variable |
| `nonlocal` | Enclosing function variable |

In [ ]:
def make_counter_broken():
    count = 0
    def counter():
        count += 1 # UnboundLocalError
        return count
    return counter

broken = make_counter_broken()

try:
    broken()
except UnboundLocalError as e:
    print(f"Error: {e}")

Error: cannot access local variable 'count' where it is not associated with a value


In [ ]:
def make_counter():
    count = 0
    def counter():
        my_count = count 
        my_count += 1
        return my_count
    return counter

broken = make_counter()

try:
    print(broken())
except UnboundLocalError as e:
    print(f"Error: {e}")

1


In [14]:
def make_counter(start: int= 0):
    count = start
    def counter():
        nonlocal count 
        count += 1
        return count
    return counter

my_new_counter = make_counter(10)

print(my_new_counter())
print(my_new_counter())
print(my_new_counter())

11
12
13


In [15]:
def make_counter():
    my_list = [0,]
    def counter():
        my_list.append(my_list[-1] + 1)
        return my_list
    return counter

my_new_counter = make_counter()

print(my_new_counter())
print(my_new_counter())
print(my_new_counter())

[0, 1]
[0, 1, 2]
[0, 1, 2, 3]


## 4.4 Closures as Lightweight Objects

Closures can replace simple classes — they're "objects with a single method."

| Approach | Best for |
|----------|----------|
| Closure | Single behavior + state |
| Class | Multiple methods + complex state |

In [16]:
def make_account(owner: str, balance: float = 0):
    """Bank Acount as closures - function sharing state"""
    current_balance = balance

    def deposit(amount: float) -> float:
        nonlocal current_balance
        current_balance += amount
        return current_balance
    
    def withdraw(amount: float) -> float:
        nonlocal current_balance
        if amount > current_balance:
            raise ValueError(f"Insufficient funds. Your ballance: {current_balance}")
        current_balance -= amount
        return current_balance
    
    def get_balance() -> float:
        return current_balance
    
    def info() -> str:
        return f"{owner}: ${current_balance}"
    
    return deposit, withdraw, get_balance, info





In [20]:
deposit, withdraw, get_balance, info = make_account("Alice", 100)
print(info())
deposit(325.50)
print(info())
withdraw(35.6)
print(info())

Alice: $100
Alice: $425.5
Alice: $389.9


## 4.5 `functools.partial` — Closure Alternative

`partial` is the stdlib way to create closures that pre-fill arguments.
When your closure just forwards args, `partial` is cleaner.

In [22]:
from functools import partial

# Without partial — manual closure
def make_multiplier_closure(factor):
    def multiplier(x):
        return x * factor
    return multiplier
 
double_c = make_multiplier_closure(2)

# ----------------
def multiply(x: float, factor: float) -> float:
    return x * factor

double_p = partial(multiply, factor= 2)
triple_p = partial(multiply, factor= 3)

print(f"double_c(5) = {double_c(5)}")
print(f"double_p(5) = {double_p(5)}")
print(f"triple_p(5) = {triple_p(5)}")

double_c(5) = 10
double_p(5) = 10
triple_p(5) = 15


In [ ]:
import json

pretty_jason = partial(json.dumps, indent= 2, sort_keys= True) # **kwargs: indent= 2, sort_keys= True
compact_json = partial(json.dumps, separators= (', ', ':'))

data = {"name":"Alice", "age": 30, "active": True}

In [24]:
print("Pretty json:")
print(pretty_jason(data))

Pretty json:
{
  "active": true,
  "age": 30,
  "name": "Alice"
}


In [27]:
print("Compact json:")
print(compact_json(data))

Compact json:
{"name":"Alice", "age":30, "active":true}


### Without partial would made something like this below.

In [28]:
import json
 
def make_json_dumper(**fixed_kwargs):
    def dumper(obj):
        return json.dumps(obj, **fixed_kwargs)
    return dumper
 
pretty_json = make_json_dumper(indent=2, sort_keys=True)
compact_json = make_json_dumper(separators=(',', ':'))
 
data = {"name": "Alice", "age": 30, "active": True}
 
print("Pretty:")
print(pretty_json(data))
 
print(f"\nCompact: {compact_json(data)}")

Pretty:
{
  "active": true,
  "age": 30,
  "name": "Alice"
}

Compact: {"name":"Alice","age":30,"active":true}


## 4.6 Under the Hood: `__closure__`

Python stores captured variables in a special `__closure__` attribute.

In [29]:
def make_adder(n: int):
    def adder(x: int):
        return x + n
    return adder
 
add_five = make_adder(5)
 
# Inspect the closure
print(f"Has closure: {add_five.__closure__ is not None}")
print(f"Closure cells: {add_five.__closure__}")
print(f"Captured value: {add_five.__closure__[0].cell_contents}")
 
# Regular functions don't have closures
def regular_func(x):
    return x + 1
 
print(f"\nRegular function closure: {regular_func.__closure__}")

Has closure: True
Closure cells: (<cell at 0x77e594a7b9d0: int object at 0xb36128>,)
Captured value: 5

Regular function closure: None


## 4.9 Exercises

**Exercise 1:** Write a `make_accumulator(start=0)` closure that:
- Each call **adds** the argument to a running total
- Returns the current total
- Example: `acc = make_accumulator(10)` → `acc(5)` = 15 → `acc(3)` = 18

In [ ]:
# Your solution here


**Exercise 2:** Write a `make_password_checker(min_length, require_digit)` closure factory that returns a validation function.
The validator should return `(True, "OK")` or `(False, "reason")`.

In [ ]:
# Your solution here


## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | Closure | Inner function that remembers enclosing scope variables |
| 2 | Factory functions | `make_X()` pattern creates families of functions |
| 3 | `nonlocal` | Required to **modify** (not just read) captured variables |
| 4 | Without `nonlocal` | `count += 1` → `UnboundLocalError` (treated as local) |
| 5 | Closures vs Classes | Closures = lightweight objects with hidden state |
| 6 | `functools.partial` | Stdlib shortcut for closures that just pre-fill args |
| 7 | `__closure__` | Inspect captured values: `func.__closure__[0].cell_contents` |
| 8 | Late binding ⚠️ | Loop closures share the same variable — use default args to fix |
| 9 | Decorators = closures | `wrapper` closes over `func` from the decorator |
| 10 | Java comparison | Like lambda captures, but more powerful with `nonlocal` |